# 01 — Data Profiling and Quality Assessment

## AI-Powered Customer Support Analytics

This notebook profiles the raw customer support ticket dataset and documents data quality issues before any cleaning or feature engineering is performed.

The objective is to understand the dataset structure, assess data completeness and consistency, inspect categorical and text fields, and identify issues that may affect downstream SQL, NLP, machine learning, and Power BI analysis.

## Objective

The objectives of this notebook are to:

1. Understand the dataset structure and data types.
2. Measure missing values and duplicate records.
3. Examine the distribution of key categorical variables.
4. Inspect ticket descriptions for data quality and templated-text issues.
5. Identify relationships between ticket status and resolution-related fields.
6. Document all identified data quality issues.
7. Assess the suitability of available fields for downstream analytics and modeling.

> **Important:** No cleaning, transformation, feature engineering, or data removal is performed during the initial profiling stage.

## 1. Setup and Data Loading

This section imports the required libraries and loads the raw dataset from the project's `data/raw/` directory.

The raw dataset is treated as an immutable source and will not be modified in this notebook during the profiling stage.

In [1]:
import pandas as pd
import numpy as np

In [2]:
file_path = "../data/raw/customer_support_tickets.csv"

df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")

Dataset shape: (8469, 17)


## 2. Dataset Overview

This section examines the dataset dimensions, column names, sample records, and data types to establish an initial understanding of the available fields.

In [3]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [4]:
print(df.columns.tolist())

['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Ticket ID                     8469 non-null   int64  
 1   Customer Name                 8469 non-null   str    
 2   Customer Email                8469 non-null   str    
 3   Customer Age                  8469 non-null   int64  
 4   Customer Gender               8469 non-null   str    
 5   Product Purchased             8469 non-null   str    
 6   Date of Purchase              8469 non-null   str    
 7   Ticket Type                   8469 non-null   str    
 8   Ticket Subject                8469 non-null   str    
 9   Ticket Description            8469 non-null   str    
 10  Ticket Status                 8469 non-null   str    
 11  Resolution                    2769 non-null   str    
 12  Ticket Priority               8469 non-null   str    
 13  Ticket Channel

## 3. Data Quality Assessment

This section evaluates data completeness and record duplication in the raw dataset.

The analysis is descriptive only. No missing values or duplicate records are modified or removed at this stage.

### 3.1 Missing Values

Missing values are measured for every column to identify fields that may require cleaning, exclusion, or business-rule-based handling in later stages of the project.

In [6]:
# Calculate missing-value counts and percentages to assess data completeness
# before defining the appropriate cleaning rules.
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_percentage", ascending=False)

missing_summary

,missing_count,missing_percentage
Resolution,5700,67.30
Time to Resolution,5700,67.30
Customer Satisfaction Rating,5700,67.30
First Response Time,2819,33.29


### Findings

Four columns contain missing values:

| Column | Missing Values | Missing Percentage |
|---|---:|---:|
| `Resolution` | 5,700 | 67.30% |
| `Time to Resolution` | 5,700 | 67.30% |
| `Customer Satisfaction Rating` | 5,700 | 67.30% |
| `First Response Time` | 2,819 | 33.29% |

`Resolution`, `Time to Resolution`, and `Customer Satisfaction Rating` have identical missing-value counts. This pattern will be investigated alongside `Ticket Status` before defining any cleaning or imputation rules.

The raw dataset is not modified during this profiling stage.

### 3.2 Duplicate Records

Duplicate records are assessed at both the full-row level and the ticket identifier level.

Checking `Ticket ID` separately is important because a dataset may contain non-identical duplicate rows that still represent duplicate tickets.

In [7]:
# Check for completely duplicated rows and duplicated ticket identifiers
# to distinguish repeated records from potential duplicate business entities.
full_row_duplicates = df.duplicated().sum()
ticket_id_duplicates = df["Ticket ID"].duplicated().sum()

print(f"Fully duplicated rows: {full_row_duplicates}")
print(f"Duplicated Ticket IDs: {ticket_id_duplicates}")

Fully duplicated rows: 0
Duplicated Ticket IDs: 0


### Findings

No fully duplicated rows were identified, and all `Ticket ID` values are unique.

Therefore, no duplicate-removal action is currently required. This conclusion applies to the raw dataset as loaded for analysis.

### 3.3 Missingness by Ticket Status

Resolution-related fields may be populated only for specific ticket statuses. This analysis examines the relationship between `Ticket Status` and missing values in key operational fields before any data handling rules are defined.

In [8]:
# Examine whether missingness in operational fields is associated
# with the current status of a support ticket.
status_missingness = (
    df.groupby("Ticket Status")
    .agg(
        ticket_count=("Ticket ID", "count"),
        missing_resolution=("Resolution", lambda x: x.isna().sum()),
        missing_first_response=("First Response Time", lambda x: x.isna().sum()),
        missing_time_to_resolution=("Time to Resolution", lambda x: x.isna().sum()),
        missing_csat=("Customer Satisfaction Rating", lambda x: x.isna().sum())
    )
)

status_missingness

,ticket_count,missing_resolution,missing_first_response,missing_time_to_resolution,missing_csat
Ticket Status,,,,,
Closed,2769,0,0,0,0
Open,2819,2819,2819,2819,2819
Pending Customer Response,2881,2881,0,2881,2881


### Findings

The missing-value patterns are fully explained by `Ticket Status`.

- `Resolution`, `Time to Resolution`, and `Customer Satisfaction Rating` are populated only for `Closed` tickets and are missing for all `Open` and `Pending Customer Response` tickets.
- `First Response Time` is missing only for `Open` tickets and is available for all `Closed` and `Pending Customer Response` tickets.

These values appear to represent operational states rather than random missing data. Therefore, no statistical imputation rule should be applied without considering the business meaning of each field.

The appropriate handling strategy will be defined during the data-cleaning and feature-engineering stage.

### 3.4 Categorical Variable Distributions

This section examines the distribution of key categorical fields to understand ticket composition and identify potential inconsistencies, unexpected labels, or highly imbalanced categories.

In [9]:
# Review category counts and proportions for key business dimensions.
categorical_columns = [
    "Ticket Type",
    "Ticket Status",
    "Ticket Priority",
    "Ticket Channel",
    "Product Purchased",
    "Customer Gender"
]

for column in categorical_columns:
    print(f"\n{'=' * 60}")
    print(f"{column}")
    print('=' * 60)
    
    distribution = pd.DataFrame({
        "count": df[column].value_counts(dropna=False),
        "percentage": (
            df[column].value_counts(normalize=True, dropna=False) * 100
        ).round(2)
    })
    
    print(distribution)


Ticket Type
                      count  percentage
Ticket Type                            
Refund request         1752       20.69
Technical issue        1747       20.63
Cancellation request   1695       20.01
Product inquiry        1641       19.38
Billing inquiry        1634       19.29

Ticket Status
                           count  percentage
Ticket Status                               
Pending Customer Response   2881       34.02
Open                        2819       33.29
Closed                      2769       32.70

Ticket Priority
                 count  percentage
Ticket Priority                   
Medium            2192       25.88
Critical          2129       25.14
High              2085       24.62
Low               2063       24.36

Ticket Channel
                count  percentage
Ticket Channel                   
Email            2143       25.30
Phone            2132       25.17
Social media     2121       25.04
Chat             2073       24.48

Product Purchased
 

### Findings

The key categorical variables contain consistent and well-distributed category labels.

- `Ticket Type` contains five relatively balanced categories.
- `Ticket Status` contains three categories with similar record volumes.
- `Ticket Priority` is distributed evenly across four priority levels.
- `Ticket Channel` is distributed evenly across four support channels.
- `Product Purchased` contains multiple product categories with no single product dominating the dataset.
- `Customer Gender` contains three consistently labeled categories.

No obvious category-label inconsistencies or highly sparse categories were identified during this initial review.

The final set of fields included in the analytical data model will be determined based on business relevance rather than column availability alone.

### 3.5 Operational Time Fields

This section inspects the raw format and descriptive characteristics of `First Response Time` and `Time to Resolution`.

Both fields were loaded as text, so their underlying format must be understood before any conversion or feature engineering is performed.

In [10]:
# Inspect raw examples to understand how operational time values
# are represented in the source dataset.
time_columns = [
    "First Response Time",
    "Time to Resolution"
]

for column in time_columns:
    print(f"\n{'=' * 60}")
    print(column)
    print('=' * 60)
    
    print("\nSample non-null values:")
    print(df[column].dropna().head(10).to_list())
    
    print("\nUnique non-null values:")
    print(df[column].dropna().nunique())


First Response Time

Sample non-null values:
['2023-06-01 12:15:36', '2023-06-01 16:45:38', '2023-06-01 11:14:38', '2023-06-01 07:29:40', '2023-06-01 00:12:42', '2023-06-01 10:32:47', '2023-06-01 09:25:48', '2023-06-01 17:46:49', '2023-06-01 12:05:51', '2023-06-01 19:03:53']

Unique non-null values:
5470

Time to Resolution

Sample non-null values:
['2023-06-01 18:05:38', '2023-06-01 01:57:40', '2023-06-01 19:53:42', '2023-05-31 23:51:49', '2023-06-01 09:27:51', '2023-05-31 23:08:55', '2023-06-01 15:58:59', '2023-06-01 20:29:04', '2023-06-01 06:03:17', '2023-06-01 18:23:17']

Unique non-null values:
2728


### Findings

Both `First Response Time` and `Time to Resolution` are stored as text but contain datetime-formatted values.

These fields appear to represent timestamps rather than direct duration measurements. Therefore, they cannot be interpreted as response duration or resolution duration without identifying an appropriate ticket creation or reference timestamp.

`Date of Purchase` will be inspected separately to determine whether it can support any time-based analysis. No datetime conversion or feature engineering is performed during this profiling stage.

### 3.6 Date Field Inspection

This section examines the format, range, and relationship of `Date of Purchase` to operational timestamps.

The purpose is to determine whether the available date field can support meaningful time-based analysis without making unsupported assumptions about when a support ticket was created.

In [11]:
# Inspect the raw purchase date values and overall date range.
print("Sample Date of Purchase values:")
print(df["Date of Purchase"].head(10).to_list())

print("\nUnique values:")
print(df["Date of Purchase"].nunique())

print("\nEarliest value:")
print(df["Date of Purchase"].min())

print("\nLatest value:")
print(df["Date of Purchase"].max())

Sample Date of Purchase values:
['2021-03-22', '2021-05-22', '2020-07-14', '2020-11-13', '2020-02-04', '2020-07-28', '2020-02-23', '2020-08-09', '2020-07-16', '2020-03-06']

Unique values:
730

Earliest value:
2020-01-01

Latest value:
2021-12-30


In [12]:
# Display ticket records with the available date and operational timestamp fields
# to inspect their chronological relationship.
timestamp_sample = df[
    [
        "Ticket ID",
        "Date of Purchase",
        "First Response Time",
        "Time to Resolution",
        "Ticket Status"
    ]
].head(10)

timestamp_sample

,Ticket ID,Date of Purchase,First Response Time,Time to Resolution,Ticket Status
0,1,2021-03-22,2023-06-01 12:15:36,NaN,Pending Customer Response
1,2,2021-05-22,2023-06-01 16:45:38,NaN,Pending Customer Response
2,3,2020-07-14,2023-06-01 11:14:38,2023-06-01 18:05:38,Closed
3,4,2020-11-13,2023-06-01 07:29:40,2023-06-01 01:57:40,Closed
4,5,2020-02-04,2023-06-01 00:12:42,2023-06-01 19:53:42,Closed
5,6,2020-07-28,NaN,NaN,Open
6,7,2020-02-23,NaN,NaN,Open
7,8,2020-08-09,NaN,NaN,Open
8,9,2020-07-16,2023-06-01 10:32:47,NaN,Pending Customer Response
9,10,2020-03-06,2023-06-01 09:25:48,NaN,Pending Customer Response


### Findings

`Date of Purchase` ranges from 2020-01-01 to 2021-12-30, while the available operational timestamps occur in 2023.

Therefore, `Date of Purchase` represents customer purchase history rather than ticket creation time and cannot be used as the reference point for calculating first-response or resolution durations.

The dataset does not currently provide an explicit ticket creation timestamp. As a result, duration-based operational KPIs and SLA-breach calculations cannot be derived directly from the available fields without introducing unsupported assumptions.

The chronological consistency between `First Response Time` and `Time to Resolution` will be investigated before determining whether the available timestamps can support other operational analyses.

### 3.7 Timestamp Consistency Check

For closed tickets, a resolution event would normally be expected to occur after the first response.

This section checks the chronological relationship between the available operational timestamps and identifies records where the recorded resolution timestamp occurs before the first response timestamp.

In [13]:
# Create temporary datetime representations for validation only.
# The raw source columns are not modified during this check.
first_response_dt = pd.to_datetime(
    df["First Response Time"],
    errors="coerce"
)

resolution_dt = pd.to_datetime(
    df["Time to Resolution"],
    errors="coerce"
)

# Identify records where the recorded resolution timestamp precedes
# the recorded first response timestamp.
timestamp_comparison = df[
    first_response_dt.notna() & resolution_dt.notna()
].copy()

timestamp_comparison["first_response_dt"] = first_response_dt.loc[
    timestamp_comparison.index
]

timestamp_comparison["resolution_dt"] = resolution_dt.loc[
    timestamp_comparison.index
]

invalid_timestamp_order = (
    timestamp_comparison["resolution_dt"]
    < timestamp_comparison["first_response_dt"]
)

print(f"Records with both timestamps: {len(timestamp_comparison)}")
print(
    "Records where resolution precedes first response: "
    f"{invalid_timestamp_order.sum()}"
)

print(
    "Percentage with invalid timestamp order: "
    f"{(invalid_timestamp_order.mean() * 100):.2f}%"
)

Records with both timestamps: 2769
Records where resolution precedes first response: 1365
Percentage with invalid timestamp order: 49.30%


### Findings

All 2,769 closed tickets contain both `First Response Time` and `Time to Resolution`.

However, 1,365 records (49.30%) have a recorded resolution timestamp that occurs before the recorded first-response timestamp.

Because nearly half of the records violate the expected chronological order, these fields cannot be treated as reliable inputs for deriving response or resolution durations.

Combined with the absence of an explicit ticket creation timestamp, the dataset does not support defensible duration-based SLA metrics without introducing unsupported assumptions.

The timestamp fields will therefore be retained only if they provide analytical value that does not depend on interpreting them as reliable elapsed-time measures.

## 4. Text Data Quality Assessment

This section evaluates the quality and diversity of the ticket text used for downstream NLP analysis.

The assessment focuses on missing text, duplicate descriptions, description length, and potential templated or repetitive patterns that could affect sentiment analysis and topic modeling.

In [14]:
# Measure text completeness and identify exact duplicate descriptions.
text_columns = [
    "Ticket Subject",
    "Ticket Description",
    "Resolution"
]

text_quality = pd.DataFrame({
    "non_null_count": df[text_columns].notna().sum(),
    "missing_count": df[text_columns].isna().sum(),
    "unique_values": df[text_columns].nunique(),
    "duplicate_non_null_values": [
        df[column].dropna().duplicated().sum()
        for column in text_columns
    ]
})

text_quality

,non_null_count,missing_count,unique_values,duplicate_non_null_values
Ticket Subject,8469,0,16,8453
Ticket Description,8469,0,8077,392
Resolution,2769,5700,2769,0


In [15]:
# Measure description length to understand the amount of text
# available for downstream NLP analysis.
description_length = (
    df["Ticket Description"]
    .fillna("")
    .str.len()
)

print(f"Minimum length: {description_length.min()}")
print(f"Median length: {description_length.median():.0f}")
print(f"Mean length: {description_length.mean():.2f}")
print(f"Maximum length: {description_length.max()}")

Minimum length: 151
Median length: 298
Mean length: 289.82
Maximum length: 397


In [16]:
# Inspect a sample of ticket subjects and descriptions to identify
# repetitive templates, placeholders, or other text-quality concerns.
text_sample = df[
    [
        "Ticket ID",
        "Ticket Type",
        "Ticket Subject",
        "Ticket Description"
    ]
].sample(10, random_state=42)

text_sample

,Ticket ID,Ticket Type,Ticket Subject,Ticket Description
4830,4831,Refund request,Product setup,I'm having an issue with the {product_purchase...
7075,7076,Product inquiry,Battery life,I'm having trouble connecting my {product_purc...
4715,4716,Billing inquiry,Refund request,I'm having an issue with the {product_purchase...
2022,2023,Billing inquiry,Peripheral compatibility,I'm having an issue with the {product_purchase...
676,677,Refund request,Peripheral compatibility,I'm having an issue with the {product_purchase...
2283,2284,Refund request,Refund request,I've encountered a data loss issue with my {pr...
5076,5077,Product inquiry,Software bug,I'm having an issue with the {product_purchase...
2476,2477,Refund request,Installation support,I'm having an issue with the {product_purchase...
6847,6848,Refund request,Hardware issue,I'm encountering a software bug in the {produc...
511,512,Product inquiry,Battery life,I'm having an issue with the {product_purchase...


### 4.1 Text Diversity and Template Patterns

Initial inspection suggests that ticket descriptions may follow a limited set of predefined templates with product-specific variations.

This section quantifies repeated text patterns and examines whether the available text contains sufficient semantic diversity to support downstream NLP analysis.

In [17]:
# Inspect placeholder tokens embedded in the ticket descriptions.
placeholder_pattern = r"\{[^}]+\}"

placeholders = (
    df["Ticket Description"]
    .str.extractall(f"({placeholder_pattern})")[0]
    .value_counts()
)

print("Placeholder tokens found:")
print(placeholders)

Placeholder tokens found:
0
{product_purchased}                                                                                                                                  13702
{error_message}                                                                                                                                        472
{product_id}                                                                                                                                            48
{product_name}                                                                                                                                          32
{Product_purchased}                                                                                                                                     18
                                                                                                                                                     ...  
{\n\nif ($ProductManager['price'])\n\n{\n\

In [18]:
# Replace all placeholder tokens with a common marker so that
# recurring sentence templates can be compared consistently.
normalized_templates = (
    df["Ticket Description"]
    .str.lower()
    .str.replace(placeholder_pattern, "{placeholder}", regex=True)
)

template_counts = normalized_templates.value_counts()

print(f"Distinct normalized templates: {template_counts.nunique()}")

print("\nTop 15 normalized templates:")
template_counts.head(15)

Distinct normalized templates: 16

Top 15 normalized templates:


Ticket Description
i'm having an issue with the {placeholder}. please assist. i need assistance as soon as possible because it's affecting my work and productivity.                                                   25
i'm having an issue with the {placeholder}. please assist. i'm concerned about the security of my {placeholder} and would like to ensure that my data is safe.                                      25
i'm having an issue with the {placeholder}. please assist. this problem started occurring after the recent software update. i haven't made any other changes to the device.                         25
i'm having an issue with the {placeholder}. please assist. i've noticed a peculiar error message popping up on my {placeholder} screen. it says '{placeholder}'. what does it mean?                 24
i'm having an issue with the {placeholder}. please assist. i've already contacted customer support multiple times, but the issue remains unresolved.                                     

In [19]:
# Count the number of distinct normalized text templates.
print(f"Distinct normalized templates: {template_counts.shape[0]}")

print("\nTop 15 normalized templates:")
template_counts.head(15)

Distinct normalized templates: 8074

Top 15 normalized templates:


Ticket Description
i'm having an issue with the {placeholder}. please assist. i need assistance as soon as possible because it's affecting my work and productivity.                                                   25
i'm having an issue with the {placeholder}. please assist. i'm concerned about the security of my {placeholder} and would like to ensure that my data is safe.                                      25
i'm having an issue with the {placeholder}. please assist. this problem started occurring after the recent software update. i haven't made any other changes to the device.                         25
i'm having an issue with the {placeholder}. please assist. i've noticed a peculiar error message popping up on my {placeholder} screen. it says '{placeholder}'. what does it mean?                 24
i'm having an issue with the {placeholder}. please assist. i've already contacted customer support multiple times, but the issue remains unresolved.                                     

### 4.2 Placeholder and Text Artifact Assessment

This section measures the prevalence of placeholder-like and malformed text patterns within ticket descriptions.

The objective is to identify text artifacts that may require preprocessing before downstream sentiment analysis or topic modeling.

In [20]:
# Measure how many ticket descriptions contain brace-delimited text artifacts.
brace_pattern = r"\{[^}]*\}"

contains_brace_pattern = (
    df["Ticket Description"]
    .str.contains(brace_pattern, regex=True, na=False)
)

print(f"Descriptions containing brace-based text: {contains_brace_pattern.sum()}")
print(
    "Percentage of descriptions containing brace-based text: "
    f"{contains_brace_pattern.mean() * 100:.2f}%"
)

Descriptions containing brace-based text: 8469
Percentage of descriptions containing brace-based text: 100.00%


In [21]:
# Inspect representative descriptions containing brace-based artifacts.
artifact_sample = df.loc[
    contains_brace_pattern,
    [
        "Ticket ID",
        "Ticket Subject",
        "Ticket Description"
    ]
].sample(10, random_state=42)

artifact_sample

,Ticket ID,Ticket Subject,Ticket Description
4830,4831,Product setup,I'm having an issue with the {product_purchase...
7075,7076,Battery life,I'm having trouble connecting my {product_purc...
4715,4716,Refund request,I'm having an issue with the {product_purchase...
2022,2023,Peripheral compatibility,I'm having an issue with the {product_purchase...
676,677,Peripheral compatibility,I'm having an issue with the {product_purchase...
2283,2284,Refund request,I've encountered a data loss issue with my {pr...
5076,5077,Software bug,I'm having an issue with the {product_purchase...
2476,2477,Installation support,I'm having an issue with the {product_purchase...
6847,6848,Hardware issue,I'm encountering a software bug in the {produc...
511,512,Battery life,I'm having an issue with the {product_purchase...


### Findings

All 8,469 ticket descriptions contain brace-based placeholder or template artifacts.

The artifacts include recurring placeholders such as `{product_purchased}` and `{error_message}`, along with a smaller number of malformed or noisy brace-delimited strings.

Because the surrounding natural-language text still contains issue context, urgency, troubleshooting history, and customer concerns, the ticket descriptions remain suitable for NLP analysis after targeted preprocessing.

A cleaned text field will be created for downstream modeling while preserving the original ticket description as the raw source field.

## 5. Structured Issue Taxonomy Assessment

The dataset contains both broad ticket categories and more specific ticket subjects.

This section examines the relationship between `Ticket Type` and `Ticket Subject` to understand the existing issue taxonomy and determine what additional analytical value topic modeling could provide.

In [22]:
# Examine the number of unique issue categories available at each level.
print(f"Unique Ticket Types: {df['Ticket Type'].nunique()}")
print(f"Unique Ticket Subjects: {df['Ticket Subject'].nunique()}")

print("\nTicket Types:")
print(sorted(df["Ticket Type"].unique()))

print("\nTicket Subjects:")
print(sorted(df["Ticket Subject"].unique()))

Unique Ticket Types: 5
Unique Ticket Subjects: 16

Ticket Types:
['Billing inquiry', 'Cancellation request', 'Product inquiry', 'Refund request', 'Technical issue']

Ticket Subjects:
['Account access', 'Battery life', 'Cancellation request', 'Data loss', 'Delivery problem', 'Display issue', 'Hardware issue', 'Installation support', 'Network problem', 'Payment issue', 'Peripheral compatibility', 'Product compatibility', 'Product recommendation', 'Product setup', 'Refund request', 'Software bug']


In [23]:
# Create a cross-tabulation to examine how specific ticket subjects
# are distributed across broader ticket types.
ticket_taxonomy = pd.crosstab(
    df["Ticket Type"],
    df["Ticket Subject"]
)

ticket_taxonomy

Ticket Subject,Account access,Battery life,Cancellation request,Data loss,Delivery problem,Display issue,Hardware issue,Installation support,Network problem,Payment issue,Peripheral compatibility,Product compatibility,Product recommendation,Product setup,Refund request,Software bug
Ticket Type,,,,,,,,,,,,,,,,
Billing inquiry,103,106,82,89,115,91,100,108,95,107,94,123,95,104,100,122
Cancellation request,92,104,103,115,114,103,109,99,102,113,102,97,98,108,126,110
Product inquiry,107,101,85,91,109,80,106,99,113,91,93,121,111,100,122,112
Refund request,108,119,109,97,107,99,129,119,107,104,106,107,106,104,119,112
Technical issue,99,112,108,99,116,105,103,105,122,111,101,119,107,113,109,118


## 6. Numeric Data Quality Assessment

This section examines the numeric fields available in the dataset to identify valid ranges, unusual values, and potential analytical limitations.

The assessment focuses on customer age and customer satisfaction ratings.

In [24]:
# Generate descriptive statistics for the dataset's numeric fields.
numeric_columns = [
    "Customer Age",
    "Customer Satisfaction Rating"
]

df[numeric_columns].describe()

,Customer Age,Customer Satisfaction Rating
count,8469.000000,2769.000000
mean,44.026804,2.991333
std,15.296112,1.407016
min,18.000000,1.000000
25%,31.000000,2.000000
50%,44.000000,3.000000
75%,57.000000,4.000000
max,70.000000,5.000000


In [25]:
# Examine the distribution of customer age.
age_summary = (
    df["Customer Age"]
    .value_counts()
    .sort_index()
)

print(f"Minimum customer age: {df['Customer Age'].min()}")
print(f"Maximum customer age: {df['Customer Age'].max()}")

age_summary

Minimum customer age: 18
Maximum customer age: 70


Customer Age
18    163
19    169
20    173
21    162
22    146
23    145
24    180
25    147
26    152
27    180
28    146
29    165
30    163
31    138
32    152
33    171
34    177
35    150
36    145
37    156
38    157
39    147
40    148
41    153
42    170
43    153
44    173
45    159
46    171
47    137
48    172
49    149
50    155
51    165
52    186
53    168
54    166
55    164
56    182
57    147
58    159
59    177
60    161
61    156
62    158
63    171
64    141
65    167
66    140
67    168
68    144
69    169
70    156
Name: count, dtype: int64

In [26]:
# Examine the distribution of customer satisfaction ratings.
csat_distribution = (
    df["Customer Satisfaction Rating"]
    .value_counts(dropna=False)
    .sort_index()
)

csat_distribution

Customer Satisfaction Rating
1.0     553
2.0     549
3.0     580
4.0     543
5.0     544
NaN    5700
Name: count, dtype: int64

## 7. Business Outcome Relationship Assessment

This section examines relationships between ticket priority, ticket status, and customer satisfaction.

The objective is to identify meaningful business outcome patterns and assess whether the available fields support a defensible predictive modeling target.

In [27]:
# Examine the distribution of ticket status across priority levels.
priority_status = pd.crosstab(
    df["Ticket Priority"],
    df["Ticket Status"],
    margins=True
)

priority_status

Ticket Status,Closed,Open,Pending Customer Response,All
Ticket Priority,,,,
Critical,726,692,711,2129
High,705,704,676,2085
Low,644,696,723,2063
Medium,694,727,771,2192
All,2769,2819,2881,8469


In [28]:
# Examine customer satisfaction for closed tickets across priority levels.
closed_tickets = df[df["Ticket Status"] == "Closed"].copy()

priority_csat = (
    closed_tickets
    .groupby("Ticket Priority")["Customer Satisfaction Rating"]
    .agg(
        ticket_count="count",
        average_csat="mean",
        median_csat="median"
    )
    .sort_values("average_csat")
)

priority_csat

,ticket_count,average_csat,median_csat
Ticket Priority,,,
Critical,726,2.958678,3.0
Medium,694,2.976945,3.0
High,705,2.982979,3.0
Low,644,3.052795,3.0


In [29]:
# Create a satisfaction category for closed tickets to examine
# the distribution of low, neutral, and high satisfaction outcomes.
def categorize_csat(rating):
    if rating <= 2:
        return "Low"
    elif rating == 3:
        return "Neutral"
    else:
        return "High"

closed_tickets["CSAT Category"] = (
    closed_tickets["Customer Satisfaction Rating"]
    .apply(categorize_csat)
)

csat_priority_distribution = pd.crosstab(
    closed_tickets["Ticket Priority"],
    closed_tickets["CSAT Category"],
    normalize="index"
).round(3) * 100

csat_priority_distribution

CSAT Category,High,Low,Neutral
Ticket Priority,,,
Critical,38.4,41.0,20.5
High,39.1,41.0,19.9
Low,42.5,38.2,19.3
Medium,37.2,38.8,24.1


## 8. Predictive Target Feasibility Assessment

The original project concept included an escalation-risk classifier. However, the available dataset does not contain an explicit escalation outcome, and the timestamp fields do not support reliable SLA-based target construction.

This section evaluates customer dissatisfaction as an alternative predictive outcome using observed satisfaction ratings from closed tickets.

The objective is to determine whether a low-CSAT classification target can be defined without relying on arbitrary assumptions or information leakage.

In [30]:
# Create a binary dissatisfaction target using observed CSAT ratings.
# Ratings of 1 or 2 are classified as dissatisfied.
# Ratings of 3, 4, or 5 are classified as not dissatisfied.

closed_tickets["Dissatisfied"] = (
    closed_tickets["Customer Satisfaction Rating"] <= 2
).astype(int)

dissatisfaction_distribution = (
    closed_tickets["Dissatisfied"]
    .value_counts()
    .rename(index={
        0: "Not Dissatisfied",
        1: "Dissatisfied"
    })
)

dissatisfaction_percentage = (
    closed_tickets["Dissatisfied"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename(index={
        0: "Not Dissatisfied",
        1: "Dissatisfied"
    })
)

dissatisfaction_summary = pd.DataFrame({
    "count": dissatisfaction_distribution,
    "percentage": dissatisfaction_percentage
})

dissatisfaction_summary

,count,percentage
Dissatisfied,,
Not Dissatisfied,1667,60.2
Dissatisfied,1102,39.8


In [31]:
# Examine dissatisfaction rates across key ticket attributes.
categorical_features = [
    "Ticket Type",
    "Ticket Subject",
    "Ticket Priority",
    "Ticket Channel",
    "Product Purchased"
]

for column in categorical_features:
    print("\n" + "=" * 60)
    print(column)
    print("=" * 60)

    summary = (
        closed_tickets
        .groupby(column)["Dissatisfied"]
        .agg(
            ticket_count="count",
            dissatisfaction_rate="mean"
        )
    )

    summary["dissatisfaction_rate"] = (
        summary["dissatisfaction_rate"] * 100
    ).round(2)

    print(
        summary.sort_values(
            "dissatisfaction_rate",
            ascending=False
        )
    )


Ticket Type
                      ticket_count  dissatisfaction_rate
Ticket Type                                             
Refund request                 596                 41.61
Billing inquiry                544                 40.99
Technical issue                580                 40.34
Cancellation request           516                 38.37
Product inquiry                533                 37.34

Ticket Subject
                          ticket_count  dissatisfaction_rate
Ticket Subject                                              
Delivery problem                   178                 48.31
Network problem                    201                 43.78
Account access                     171                 43.27
Battery life                       173                 42.77
Product setup                      183                 40.44
Data loss                          147                 40.14
Software bug                       199                 39.70
Product compatibility  

## 9. Text Placeholder Replacement Assessment

Ticket descriptions contain template placeholders and other brace-based artifacts. This section evaluates whether common placeholders can be replaced using structured dataset fields before NLP processing.

The goal is to improve the quality and interpretability of text used for sentiment analysis and topic modeling while preserving the original business meaning of each ticket.

In [32]:
# Inspect the relationship between the most common placeholder
# and the Product Purchased field.

placeholder_sample = df[
    df["Ticket Description"].str.contains(
        r"\{product_purchased\}",
        case=False,
        regex=True,
        na=False
    )
][
    [
        "Ticket ID",
        "Product Purchased",
        "Ticket Subject",
        "Ticket Description"
    ]
].head(10)

placeholder_sample

,Ticket ID,Product Purchased,Ticket Subject,Ticket Description
0,1,GoPro Hero,Product setup,I'm having an issue with the {product_purchase...
1,2,LG Smart TV,Peripheral compatibility,I'm having an issue with the {product_purchase...
2,3,Dell XPS,Network problem,I'm facing a problem with my {product_purchase...
3,4,Microsoft Office,Account access,I'm having an issue with the {product_purchase...
4,5,Autodesk AutoCAD,Data loss,I'm having an issue with the {product_purchase...
5,6,Microsoft Office,Payment issue,I'm facing a problem with my {product_purchase...
6,7,Microsoft Surface,Refund request,I'm unable to access my {product_purchased} ac...
7,8,Philips Hue Lights,Battery life,I'm having an issue with the {product_purchase...
8,9,Fitbit Versa Smartwatch,Installation support,I'm having an issue with the {product_purchase...
9,10,Dyson Vacuum Cleaner,Payment issue,My {product_purchased} is making strange noise...


In [33]:
import re

# Create a temporary version of the description with common product
# placeholders replaced by the corresponding Product Purchased value.

test_description = pd.Series(
    [
        re.sub(
            r"\{product_purchased\}",
            product,
            description,
            flags=re.IGNORECASE
        )
        for description, product in zip(
            df["Ticket Description"],
            df["Product Purchased"]
        )
    ],
    index=df.index
)

replacement_preview = pd.DataFrame({
    "Product Purchased": df["Product Purchased"].head(10),
    "Original Description": df["Ticket Description"].head(10),
    "Updated Description": test_description.head(10)
})

replacement_preview

,Product Purchased,Original Description,Updated Description
0,GoPro Hero,I'm having an issue with the {product_purchase...,I'm having an issue with the GoPro Hero. Pleas...
1,LG Smart TV,I'm having an issue with the {product_purchase...,I'm having an issue with the LG Smart TV. Plea...
2,Dell XPS,I'm facing a problem with my {product_purchase...,I'm facing a problem with my Dell XPS. The Del...
3,Microsoft Office,I'm having an issue with the {product_purchase...,I'm having an issue with the Microsoft Office....
4,Autodesk AutoCAD,I'm having an issue with the {product_purchase...,I'm having an issue with the Autodesk AutoCAD....
5,Microsoft Office,I'm facing a problem with my {product_purchase...,I'm facing a problem with my Microsoft Office....
6,Microsoft Surface,I'm unable to access my {product_purchased} ac...,I'm unable to access my Microsoft Surface acco...
7,Philips Hue Lights,I'm having an issue with the {product_purchase...,I'm having an issue with the Philips Hue Light...
8,Fitbit Versa Smartwatch,I'm having an issue with the {product_purchase...,I'm having an issue with the Fitbit Versa Smar...
9,Dyson Vacuum Cleaner,My {product_purchased} is making strange noise...,My Dyson Vacuum Cleaner is making strange nois...


## Data Quality Decisions

The initial profiling identified several data quality characteristics that will guide the downstream preprocessing and analytics design.

| Data Quality Issue | Observation | Decision |
|---|---|---|
| Missing resolution data | Resolution is missing for all open and pending tickets | Preserve missing values because a resolution does not yet exist for unresolved tickets |
| Missing time to resolution | Available only for closed tickets | Preserve missing values and avoid imputing resolution timestamps |
| Missing CSAT | Available only for closed tickets | Preserve missing values because satisfaction is recorded only after ticket resolution |
| Missing first response time | Missing for all open tickets but available for closed and pending tickets | Preserve missing values and treat response availability as part of the ticket lifecycle |
| Invalid timestamp ordering | A substantial proportion of closed tickets have a resolution timestamp earlier than the first response timestamp | Do not derive response or resolution duration metrics directly from these timestamps without additional validation |
| Customer identifiers | Customer Name and Customer Email are personally identifiable fields | Exclude these fields from analytics and modeling |
| Product placeholders in ticket text | Ticket descriptions contain product placeholders such as `{product_purchased}` | Replace recognized product placeholders with the corresponding `Product Purchased` value |
| Other brace-based artifacts | Some descriptions contain malformed or unrelated brace-based text | Normalize or remove these artifacts during downstream text preprocessing |
| Synthetic text patterns | Ticket descriptions contain repeated templates | Account for templated language when interpreting NLP and topic modeling results |

These decisions establish the preprocessing rules for creating an analytics-ready ticket dataset while preserving the distinction between genuine missing values and unresolved ticket states.

## Final Feature Set

Based on the initial profiling, the following decisions define which source columns will be retained, excluded, or used to derive downstream features.

### Retained Source Features

The following fields are suitable for the analytics dataset and downstream analysis:

- `Ticket ID`
- `Customer Age`
- `Customer Gender`
- `Product Purchased`
- `Date of Purchase`
- `Ticket Type`
- `Ticket Subject`
- `Ticket Description`
- `Ticket Status`
- `Resolution`
- `Ticket Priority`
- `Ticket Channel`
- `First Response Time`
- `Time to Resolution`
- `Customer Satisfaction Rating`

### Excluded Features

The following fields will be excluded from analytics and modeling because they are customer identifiers and do not contribute to the business analysis:

- `Customer Name`
- `Customer Email`

### Planned Derived Features

The following features are planned for later preprocessing and modeling steps:

- Cleaned ticket text for NLP processing
- Sentiment label and sentiment confidence
- Topic or issue classification
- Customer dissatisfaction label derived from Customer Satisfaction Rating
- Customer dissatisfaction risk score generated by the supervised model

### Timestamp Limitation

The available response and resolution timestamps contain substantial ordering inconsistencies. Therefore, duration-based features will not be finalized until the timestamp quality issue is addressed during preprocessing.

### Feature Availability Principle

Predictive features will be selected based on whether they would be available at the relevant prediction point. Fields representing outcomes that occur after ticket resolution will not be used as predictors for the customer dissatisfaction risk model.

## Raw Data Staging with DuckDB

The raw customer support ticket dataset is loaded into DuckDB as a staging table. The staging layer preserves the source data structure and provides a SQL-based foundation for downstream preprocessing and analytics.

In [37]:
import duckdb

# Define the DuckDB database connection.
conn = duckdb.connect("../data/customer_support.duckdb")

# Load the raw CSV into a staging table.
conn.execute("""
    CREATE OR REPLACE TABLE stg_customer_support_tickets AS
    SELECT *
    FROM read_csv_auto("../data/raw/customer_support_tickets.csv");
""")

print("Staging table created successfully.")

Staging table created successfully.


In [38]:
# Verify the number of records loaded into the staging table.

staging_count = conn.execute("""
    SELECT COUNT(*) AS ticket_count
    FROM stg_customer_support_tickets;
""").df()

staging_count

,ticket_count
0,8469


In [39]:
# Inspect the staging table schema.

schema_check = conn.execute("""
    DESCRIBE stg_customer_support_tickets;
""").df()

schema_check

,column_name,column_type,null,key,default,extra
0,Ticket ID,BIGINT,YES,None,None,None
1,Customer Name,VARCHAR,YES,None,None,None
2,Customer Email,VARCHAR,YES,None,None,None
3,Customer Age,BIGINT,YES,None,None,None
4,Customer Gender,VARCHAR,YES,None,None,None
5,Product Purchased,VARCHAR,YES,None,None,None
6,Date of Purchase,DATE,YES,None,None,None
7,Ticket Type,VARCHAR,YES,None,None,None
8,Ticket Subject,VARCHAR,YES,None,None,None
9,Ticket Description,VARCHAR,YES,None,None,None
